# Notebook 5b — DINOv3 + DinoTXT Zero-Shot Verb Prediction

This notebook performs zero-shot verb prediction on YouCook2 using **DINOv3** as the vision
backbone, aligned with the **DinoTXT** text tower — a direct parallel to the DINOv2 DinoTXT
notebook (NB5) for controlled backbone comparison.

**Approach (identical to DINOv2 notebook):**
- Extract per-frame embeddings from each video segment using the frozen DINOv3 vision backbone
- Temporally pool frame embeddings into a single video-level representation
- For each segment, fill the `[MASK]` slot in the masked caption with every verb in the vocabulary
- Encode all candidates with the frozen DinoTXT text tower
- Pick the verb whose text embedding is most similar (cosine) to the video embedding
- No training — purely zero-shot similarity in the shared vision-language space

**Key difference from DINOv2 notebook:**
- Vision backbone is DINOv3 ViT-L/16 instead of DINOv2 ViT-L/14
- DinoTXT text tower is loaded from the DINOv3 repo (same architecture, re-trained on DINOv3 features)
- Frame preprocessing uses DINOv3's image transforms (patch size 16, different normalisation)

**2026-08-26 fixes applied (see bottom of notebook history / conversation log):**
- Standardized on `segments_verbs_timed_clean.csv` everywhere in this notebook (was
  mixing this with the unclean `segments_verbs_timed.csv`, which has 13,029 rows /
  3,287 val instead of the canonical 13,003 / 3,268 — a real discrepancy with NB5
  and NB6, which always used the clean file).
- Removed the `df_val.head(500)` slice — the zero-shot loop now runs the full
  3,268-segment validation set, matching NB6/V-JEPA2's full-val evaluation
  (previously this notebook's headline number was reported on 500 arbitrary segments).
- Removed the redundant second loop that re-evaluated on train+val combined
  (13,029 segments) for no downstream benefit — one full-val pass is now authoritative.
- The DINOv2-comparison cells now read the DINOv2 side from the *completed* full-val
  run (`processed/dinov2_zeroshot_fullval_fixed.csv`) instead of the old
  `results_checkpoint.jsonl`, which was an interrupted run (566/13,003, only 134 in
  val) — this raises the DINOv2-vs-DINOv3 common-segment N from 134 to the full
  val-set intersection.
- Fixed `FRAMES_PER_SEG=16` in the (dead — always hits the "already exists" skip)
  frame-extraction cell; it claimed consistency with NB4 but NB4 actually used
  `MAX_FRAMES=50`. Fixed to 50 so the code is correct if anyone ever reruns it from
  scratch on a fresh embeddings directory.
- Fixed a stale docstring/comment claiming the shared embedding is 1280-dim; the
  actual runtime shape (and what's used everywhere downstream) is 2048-dim.
- **Not fixed / still open**: this run still can't execute end-to-end in the current
  environment — see the weights blocker noted in the Prerequisites section below.

**Reads:**
- `annotations/segments_verbs_timed_clean.csv` — captions, verbs, segment IDs (13,003 rows, 9,735 train / 3,268 val)
- Raw video files (to extract frames on the fly) OR precomputed DINOv3 embeddings

**Produces:**
- Top-1 and Top-5 verb prediction accuracy on the full validation set
- `processed/nb5b_dinov3_dinotxt_zeroshot_results.csv`

**Prerequisites:**
1. DINOv3 repo cloned: `git clone https://github.com/facebookresearch/dinov3.git`
2. DINOv3 ViT-L/16 backbone weights downloaded (requires Meta license approval — the
   direct download URLs return 403 without a signed, approval-gated link)
3. DINOv3 DinoTXT text tower weights downloaded (same approval email includes both)

> **⚠️ Environment blocker (2026-08-26):** neither weight file exists in this
> environment (`/home/ubuntu/weights/` is empty/missing, and there is no HF token or
> cached copy anywhere on this box). `dl.fbaipublicfiles.com/dinov3/...` returns
> `403 Forbidden` without a signed URL from Meta's approval email — unlike DINOv2's
> weights, these are not publicly downloadable. This notebook cannot be executed
> until the two `.pth` files are placed back at the paths in the cell below, or the
> paths are updated to wherever they're re-obtained from.

## Setup

In [ ]:
import os, sys, json
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

## Paths — update these to match your setup

In [ ]:
# Path to the cloned DINOv3 repository
DINOV3_REPO = os.path.join(PATHS['repos'], 'dinov3')

# Paths to the downloaded weight files
# These are the URLs/paths sent to you after HuggingFace license approval
DINOV3_BACKBONE_WEIGHTS = '/home/ubuntu/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
DINOV3_DINOTXT_WEIGHTS  = '/home/ubuntu/weights/dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth'

# Where to save DINOv3 frame embeddings (separate from DINOv2 embeddings)
DINOV3_EMB_DIR = os.path.join(PATHS['embeddings'], 'dinov3')
os.makedirs(DINOV3_EMB_DIR, exist_ok=True)

print(f"DINOv3 repo       : {DINOV3_REPO}")
print(f"Backbone weights  : {DINOV3_BACKBONE_WEIGHTS}")
print(f"DinoTXT weights   : {DINOV3_DINOTXT_WEIGHTS}")
print(f"Embedding dir     : {DINOV3_EMB_DIR}")

# Sanity check — make sure weight files exist
for p, name in [(DINOV3_BACKBONE_WEIGHTS, 'backbone'), (DINOV3_DINOTXT_WEIGHTS, 'dinotxt')]:
    if os.path.exists(p):
        print(f"✓ {name} weights found")
    else:
        print(f"✗ {name} weights NOT found at {p} — update the path above")

## Load annotation data

In [ ]:
df = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_clean.csv')

# Build verb vocabulary from all extracted verbs
verb_vocab = sorted(df['extracted_verb'].unique().tolist())
n_verbs    = len(verb_vocab)

print(f"Total segments : {len(df):,}")
print(f"Verb classes   : {n_verbs}")
print(f"Columns        : {df.columns.tolist()}")
print(df.head(2))

## Load DINOv3 + DinoTXT model (fully frozen)

This uses `torch.hub.load` from the local DINOv3 repo clone, passing both the
DinoTXT text tower weights and the DINOv3 backbone weights separately.

The model architecture is identical to DINOv2 DinoTXT — same shared embedding
dimension (1280), same text tower size — but the vision head was trained to align
specifically with DINOv3 features rather than DINOv2 features.

In [ ]:
# Add DINOv3 repo to path so its internal imports resolve
if DINOV3_REPO not in sys.path:
    sys.path.insert(0, DINOV3_REPO)

print("Loading DINOv3 + DinoTXT...")
dinotxt, tokenizer = torch.hub.load(
    DINOV3_REPO,
    'dinov3_vitl16_dinotxt_tet1280d20h24l',
    source='local',
    weights=DINOV3_DINOTXT_WEIGHTS,
    backbone_weights=DINOV3_BACKBONE_WEIGHTS,
)
dinotxt = dinotxt.to(DEVICE).eval()

# Freeze all parameters — this is zero-shot, nothing trains
for p in dinotxt.parameters():
    p.requires_grad = False

total     = sum(p.numel() for p in dinotxt.parameters())
trainable = sum(p.numel() for p in dinotxt.parameters() if p.requires_grad)
print(f"Model loaded: {total/1e6:.1f}M parameters, {trainable} trainable (must be 0)")
assert trainable == 0, "Model must be fully frozen"
print("All parameters frozen ✓")

## Image preprocessing transforms

DINOv3 uses patch size 16 (vs DINOv2's patch size 14) and different normalisation
statistics for LVD-1689M pretrained models. We use DINOv3's built-in transform.

In [ ]:
from dinov3.data.transforms import make_classification_eval_transform

# DINOv3 standard eval transform — 224x224 centre crop, ImageNet normalisation
image_transform = make_classification_eval_transform(resize_size=256, crop_size=224)

print("Image transform ready ✓")
print("  resize_size : 256")
print("  crop_size   : 224")
print("  normalisation: DINOv3 LVD-1689M (ImageNet stats)")

## Extract and save DINOv3 video embeddings

For each video segment, we:
1. Load the raw video frames for that segment
2. Preprocess each frame with the DINOv3 transform
3. Run frames through `dinotxt.encode_image()` in sub-batches
4. Mean-pool frame embeddings → single segment embedding
5. Save to `embeddings/dinov3/{segment_id}.pt`

This is identical to what NB4 did for DINOv2, just using the DINOv3 vision tower.
Skip this cell if embeddings are already computed.

In [ ]:
import cv2
from PIL import Image

FRAME_BATCH_SIZE = 32   # sub-batch size for GPU memory safety on A100
FRAMES_PER_SEG   = 50   # matches NB4's MAX_FRAMES=50 (was wrongly set to 16 here —
                         # this cell is dead code in practice since embeddings already
                         # exist and it always hits the "skip" branch below, but it's
                         # fixed so a from-scratch rerun would actually match NB4)

def extract_frames_for_segment(video_path: str, start_sec: float, end_sec: float, n_frames: int = FRAMES_PER_SEG):
    """Extract n_frames uniformly from a video segment [start_sec, end_sec]."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    start_frame = int(start_sec * fps)
    end_frame   = int(end_sec   * fps)
    frame_indices = np.linspace(start_frame, end_frame - 1, n_frames, dtype=int)

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
    cap.release()
    return frames


@torch.no_grad()
def embed_segment_dinov3(video_path: str, start_sec: float, end_sec: float) -> torch.Tensor:
    """Extract DINOv3 embedding for a video segment. Returns (2048,) L2-normalised."""
    frames = extract_frames_for_segment(video_path, start_sec, end_sec)
    if len(frames) == 0:
        return None

    frame_tensors = torch.stack([image_transform(f) for f in frames])  # (N, 3, 224, 224)

    all_embeddings = []
    with torch.autocast('cuda', dtype=torch.float):
        for i in range(0, len(frame_tensors), FRAME_BATCH_SIZE):
            batch = frame_tensors[i:i + FRAME_BATCH_SIZE].to(DEVICE)
            emb   = dinotxt.encode_image(batch)  # (B, 2048)
            all_embeddings.append(emb.cpu())

    # Temporal mean pooling across frames
    video_emb = torch.cat(all_embeddings, dim=0).mean(dim=0)  # (2048,)
    video_emb = video_emb / video_emb.norm()                  # L2 normalise
    return video_emb


# Run extraction — skip segments that already have embeddings saved
VIDEO_DIR = PATHS['videos']
skipped = 0
errors  = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting DINOv3 embeddings"):
    out_path = os.path.join(DINOV3_EMB_DIR, f"{row['segment_id']}.pt")
    if os.path.exists(out_path):
        skipped += 1
        continue

    video_path = os.path.join(VIDEO_DIR, f"{row['video_id']}.mp4")
    if not os.path.exists(video_path):
        errors += 1
        continue

    try:
        emb = embed_segment_dinov3(video_path, row['start_time'], row['end_time'])
        if emb is not None:
            torch.save(emb, out_path)
    except Exception as e:
        errors += 1

saved = len([f for f in os.listdir(DINOV3_EMB_DIR) if f.endswith('.pt')])
print(f"Embeddings saved : {saved:,}")
print(f"Skipped (exist)  : {skipped:,}")
print(f"Errors           : {errors:,}")

## Precompute verb text embeddings

Identical to the DINOv2 DinoTXT notebook — for each segment we fill `[MASK]` with
every verb in the vocabulary using that segment's actual masked caption context,
then encode with the DinoTXT text tower.

The text tower architecture and shared embedding dimension (1280) are the same
between DINOv2-DinoTXT and DINOv3-DinoTXT, so this function is directly comparable.

In [ ]:
CONTEXT_LENGTH = 77  # CLIP tokenizer max length

@torch.no_grad()
def get_text_embeddings_for_segment(masked_caption: str) -> torch.Tensor:
    """
    Given a masked caption like '[MASK] the onions into small pieces',
    replace [MASK] with every verb in vocab and encode all candidates.
    Returns tensor of shape (n_verbs, 2048), L2-normalised.
    """
    caption_context = masked_caption.replace('[MASK]', '{}')
    verb_texts      = [caption_context.format(v) for v in verb_vocab]

    with torch.autocast('cuda', dtype=torch.float):
        tokens        = tokenizer.tokenize(verb_texts).to(DEVICE)
        text_features = dinotxt.encode_text(tokens)           # (n_verbs, 2048)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features


# Quick sanity check
test_caption = df['masked_caption'].iloc[0]
test_emb     = get_text_embeddings_for_segment(test_caption)
print(f"Test caption          : {test_caption}")
print(f"Text embeddings shape : {test_emb.shape}  (expected: ({n_verbs}, 2048))")
print(f"L2 norm               : {test_emb.norm(dim=-1).mean():.4f}  (expected: ~1.0)")

## Zero-shot verb prediction

For each segment:
1. Load the precomputed DINOv3 video embedding
2. Encode all verb candidates in the caption context using DinoTXT text tower
3. Pick the verb with highest cosine similarity to the video embedding
4. Record Top-1 and Top-5 correctness

This loop is **identical** to the DINOv2 DinoTXT notebook — only the embedding
directory changes, making results directly comparable.

In [ ]:
import pandas as pd

df = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_clean.csv')

verb_vocab = sorted(df['extracted_verb'].unique())
label2id   = {v: i for i, v in enumerate(verb_vocab)}
id2label   = {i: v for v, i in label2id.items()}
n_verbs    = len(verb_vocab)
df['verb_label'] = df['extracted_verb'].map(label2id)

df_train = df[df['subset'] == 'training'].reset_index(drop=True)
df_val   = df[df['subset'] == 'validation'].reset_index(drop=True)

print(f"Train : {len(df_train):,}")
print(f"Val   : {len(df_val):,}")

In [ ]:
results = []
missing = 0

# Full validation set (previously .head(500) — an arbitrary slice that made this
# notebook's headline number non-comparable to NB6's full-3,268-segment number)
df_sample = df_val

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Zero-shot prediction (full val)"):
    emb_path = os.path.join(DINOV3_EMB_DIR, f"{row['segment_id']}.pt")

    if not os.path.exists(emb_path):
        missing += 1
        continue

    video_emb = torch.load(emb_path, map_location=DEVICE)  # (2048,)
    video_emb = video_emb / video_emb.norm()

    text_embs = get_text_embeddings_for_segment(row['masked_caption'])  # (n_verbs, 2048)

    similarity   = text_embs @ video_emb
    top5_indices = similarity.topk(5).indices.tolist()
    top1_verb    = verb_vocab[top5_indices[0]]
    ground_truth = row['extracted_verb']
    gt_idx       = verb_vocab.index(ground_truth) if ground_truth in verb_vocab else -1

    results.append({
        'segment_id'    : row['segment_id'],
        'subset'        : row['subset'],
        'ground_truth'  : ground_truth,
        'top1_pred'     : top1_verb,
        'top5_preds'    : [verb_vocab[i] for i in top5_indices],
        'top1_correct'  : top1_verb == ground_truth,
        'top5_correct'  : gt_idx in top5_indices,
        'masked_caption': row['masked_caption'],
    })

results_df = pd.DataFrame(results)
top1 = results_df['top1_correct'].mean()
top5 = results_df['top5_correct'].mean()

print(f"Val segments        : {len(df_sample):,}")
print(f"Predicted          : {len(results):,}")
print(f"Missing embeddings : {missing:,}")
print(f"\nVal Top-1          : {top1:.4f}  ({top1*100:.2f}%)")
print(f"Val Top-5          : {top5:.4f}  ({top5*100:.2f}%)")
print(f"Random baseline    : {1/len(verb_vocab):.4f}  ({100/len(verb_vocab):.2f}%)")

In [ ]:
print("=== First 20 segment predictions ===\n")
for i, row in results_df.head(20).iterrows():
    top1_mark = "✓" if row['top1_correct'] else "✗"
    top5_mark = "✓ in top-5" if row['top5_correct'] else "✗"
    print(f"Segment  : {row['segment_id']}")
    print(f"Caption  : {row['masked_caption']}")
    print(f"GT verb  : {row['ground_truth']}")
    print(f"Top-1    : {row['top1_pred']}  {top1_mark}")
    print(f"Top-5    : {row['top5_preds']}  {top5_mark}")
    print()

## Results

In [ ]:
results_df = pd.DataFrame(results)

# Overall accuracy
top1_overall = results_df['top1_correct'].mean()
top5_overall = results_df['top5_correct'].mean()
print(f"Overall Top-1 : {top1_overall:.4f}  ({top1_overall*100:.2f}%)")
print(f"Overall Top-5 : {top5_overall:.4f}  ({top5_overall*100:.2f}%)")

print()
for subset in ['training', 'validation']:
    sub = results_df[results_df['subset'] == subset]
    if len(sub) == 0:
        continue
    t1 = sub['top1_correct'].mean()
    t5 = sub['top5_correct'].mean()
    print(f"{subset:12s} — Top-1: {t1:.4f} ({t1*100:.2f}%)  Top-5: {t5:.4f} ({t5*100:.2f}%)")

## Side-by-side comparison with DINOv2

Load the DINOv2 DinoTXT results from NB5 and compare directly.

In [ ]:
import os
import json
import subprocess

# Search common locations
search_paths = [
    '/home/ubuntu/youcook2-videos',
    '/home/ubuntu',
    PATHS.get('processed', ''),
    PATHS.get('outputs', ''),
]

print("=== Searching for results_checkpoint.jsonl ===")
for base in search_paths:
    if not base:
        continue
    for root, dirs, files in os.walk(base):
        for f in files:
            if 'checkpoint' in f.lower() or 'results' in f.lower():
                full = os.path.join(root, f)
                size = os.path.getsize(full) / 1e6
                print(f"  {full}  ({size:.1f} MB)")

In [ ]:
import pandas as pd

# Fixed: previously loaded from results_checkpoint.jsonl, an interrupted DINOv2 run
# (566/13,003 segments, only 134 landing in val) which capped every downstream
# common-segment comparison at N=134. Now loads the completed full-val DINOv2 run
# (processed/dinov2_zeroshot_fullval_fixed.csv, all 3,268 val segments), so the
# common-segment N below is the full val-set intersection instead.
DINOV2_FULLVAL_PATH = os.path.join(PATHS['processed'], 'dinov2_zeroshot_fullval_fixed.csv')

if not os.path.exists(DINOV2_FULLVAL_PATH):
    raise FileNotFoundError(
        f"{DINOV2_FULLVAL_PATH} not found — run "
        "notebooks/scripts/dinov2_zeroshot_fullval_fixed.py first."
    )

dinov2_df = pd.read_csv(DINOV2_FULLVAL_PATH)
print(f"DINOv2 total segments processed : {len(dinov2_df):,}")
print(f"DINOv3 total segments processed : {len(results_df):,}")

# Find common segment_ids
common_ids = set(dinov2_df['segment_id']) & set(results_df['segment_id'])
print(f"Common segments                 : {len(common_ids):,}")

# Filter to common segments only
dinov2_common = dinov2_df[dinov2_df['segment_id'].isin(common_ids)].set_index('segment_id')
dinov3_common = results_df[results_df['segment_id'].isin(common_ids)].set_index('segment_id')

print(f"\n=== Comparison on {len(common_ids):,} shared segments ===\n")
print(f"{'Model':<22} {'Top-1':>10} {'Top-5':>10} {'N':>6}")
print("=" * 55)

for subset in ['training', 'validation']:
    sub_v2 = dinov2_common[dinov2_common['subset'] == subset] if 'subset' in dinov2_common.columns else dinov2_common
    sub_v3 = dinov3_common[dinov3_common['subset'] == subset] if 'subset' in dinov3_common.columns else dinov3_common

    if len(sub_v2) > 0 or len(sub_v3) > 0:
        print(f"\n  Subset: {subset}")
        print("-" * 55)
    if len(sub_v2) > 0:
        t1 = sub_v2['top1_correct'].mean()
        t5 = sub_v2['top5_correct'].mean()
        print(f"  {'DINOv2 + DinoTXT':<20} {t1*100:>9.2f}% {t5*100:>9.2f}%  {len(sub_v2):>5}")
    if len(sub_v3) > 0:
        t1 = sub_v3['top1_correct'].mean()
        t5 = sub_v3['top5_correct'].mean()
        print(f"  {'DINOv3 + DinoTXT':<20} {t1*100:>9.2f}% {t5*100:>9.2f}%  {len(sub_v3):>5}")

print("\n" + "=" * 55)
print(f"  Random baseline : {100/len(verb_vocab):.2f}%")
print("=" * 55)

In [ ]:
sample_emb = torch.load(
    os.path.join(DINOV3_EMB_DIR, os.listdir(DINOV3_EMB_DIR)[0]),
    map_location='cpu'
)
print(f"DINOv3 video embedding dim : {sample_emb.shape}")
print(f"Text embedding dim         : {text_embs.shape}")

In [ ]:
CKPT_DIR = PATHS.get('checkpoints', '/home/ubuntu/youcook2-videos/checkpoints')

> **⚠️ Known bug in the next cell (`e9d39284`):** it loads the V-JEPA2 projection
> checkpoint onto `dinotxt`, which at this point in the notebook is the **DINOv3**-based
> model — but the checkpoint was trained against the **DINOv2**-based text tower. This
> produces a meaningless result (Top-1 ≈0.00%, Top-5 ≈0.75% on N=134). **Do not cite
> that number.** The correct evaluation — same checkpoint, loaded onto a freshly
> constructed DINOv2-based `dinotxt_vj` — is a few cells further down (`1a532c8c`) and
> gets Top-1 ≈38.81% / Top-5 ≈64.93% on the same 134 segments. Left in place
> deliberately as a worked example of the bug rather than silently deleted.

In [ ]:
import torch.nn as nn

# Replace projection to match V-JEPA2 checkpoint shape (1280 → 1024)
dinotxt.text_model.head.linear_projection = nn.Linear(1280, 1024, bias=False).to(DEVICE)

# Now load the checkpoint
ckpt = torch.load(
    os.path.join(CKPT_DIR, 'vjepa2_dinotxt_v2_cont_epoch5.pt'),
    map_location=DEVICE
)
dinotxt.text_model.head.linear_projection.load_state_dict(ckpt)
dinotxt.eval()

# Freeze everything
for param in dinotxt.parameters():
    param.requires_grad = False

print(f"Loaded V-JEPA2 checkpoint ✓")
print(f"Projection shape: {dinotxt.text_model.head.linear_projection.weight.shape}")

In [ ]:
import torch.nn.functional as F

# ── Load best V-JEPA2 checkpoint ──────────────────────────────────────────────
ckpt = torch.load(
    os.path.join(CKPT_DIR, 'vjepa2_dinotxt_v2_cont_epoch5.pt'),
    map_location=DEVICE
)
dinotxt.text_model.head.linear_projection.load_state_dict(ckpt)
dinotxt.eval()
print("Loaded best V-JEPA2 checkpoint\n")

# ── Use the same common segments identified by DINOv2 and DINOv3 ──────────────
common_segment_ids = common_ids  # from the comparison cell above

# Filter df_val to only common segments
df_common = df_val[df_val['segment_id'].isin(common_segment_ids)].reset_index(drop=True)
print(f"Common segments to evaluate : {len(df_common):,}")

VJEPA2_EMB_DIR = os.path.join(PATHS['embeddings'], 'vjepa2_dinotxt')

vjepa2_results = []
missing = 0

with torch.no_grad():
    for _, row in tqdm(df_common.iterrows(), total=len(df_common), desc="V-JEPA2 prediction"):
        emb_path = os.path.join(VJEPA2_EMB_DIR, f"{row['segment_id']}.pt")
        if not os.path.exists(emb_path):
            missing += 1
            continue

        video_emb = torch.load(emb_path, map_location=DEVICE)
        video_emb = F.normalize(video_emb.unsqueeze(0), dim=-1)

        CHUNK = 64
        all_embs = []
        for i in range(0, len(verb_vocab), CHUNK):
            tokens = tokenizer.tokenize(verb_vocab[i:i+CHUNK]).to(DEVICE)
            embs   = dinotxt.encode_text(tokens, normalize=True)
            all_embs.append(embs)
        verb_embs = torch.cat(all_embs, dim=0)  # (282, 1024)

        sim          = (dinotxt.logit_scale.exp() * video_emb @ verb_embs.T).squeeze(0)
        top5_indices = sim.topk(5).indices.tolist()
        top1_verb    = verb_vocab[top5_indices[0]]
        top5_verbs   = [verb_vocab[i] for i in top5_indices]
        gt_verb      = row['extracted_verb']
        gt_idx       = label2id.get(gt_verb, -1)

        vjepa2_results.append({
            'segment_id'  : row['segment_id'],
            'subset'      : row['subset'],
            'ground_truth': gt_verb,
            'top1_pred'   : top1_verb,
            'top5_preds'  : top5_verbs,
            'top1_correct': top1_verb == gt_verb,
            'top5_correct': gt_idx in top5_indices,
            'masked_caption': row['masked_caption'],
        })

vjepa2_df = pd.DataFrame(vjepa2_results)

# ── Three-way comparison on common segments ───────────────────────────────────
print(f"\n=== Three-way comparison on {len(common_segment_ids)} common segments ===\n")
print(f"{'Model':<25} {'Top-1':>10} {'Top-5':>10} {'N':>6}")
print("=" * 58)

# DINOv2
sub_v2 = dinov2_common[dinov2_common['subset'] == 'validation']
if len(sub_v2) > 0:
    print(f"  {'DINOv2 + DinoTXT':<23} {sub_v2['top1_correct'].mean()*100:>9.2f}% {sub_v2['top5_correct'].mean()*100:>9.2f}%  {len(sub_v2):>5}")

# DINOv3
sub_v3 = dinov3_common[dinov3_common['subset'] == 'validation']
if len(sub_v3) > 0:
    print(f"  {'DINOv3 + DinoTXT':<23} {sub_v3['top1_correct'].mean()*100:>9.2f}% {sub_v3['top5_correct'].mean()*100:>9.2f}%  {len(sub_v3):>5}")

# V-JEPA2
sub_vj = vjepa2_df[vjepa2_df['subset'] == 'validation']
if len(sub_vj) > 0:
    print(f"  {'V-JEPA2 + DinoTXT':<23} {sub_vj['top1_correct'].mean()*100:>9.2f}% {sub_vj['top5_correct'].mean()*100:>9.2f}%  {len(sub_vj):>5}")

print("=" * 58)
print(f"  {'Random baseline':<23} {100/len(verb_vocab):>9.2f}% {500/len(verb_vocab):>9.2f}%")
print("=" * 58)
print(f"\nMissing V-JEPA2 embeddings : {missing}")

# ── Per-segment comparison for first 10 common segments ───────────────────────
print(f"\n=== Per-segment comparison (first 10) ===\n")
for seg_id in list(common_segment_ids)[:10]:
    row_vj = vjepa2_df[vjepa2_df['segment_id'] == seg_id]
    row_v2 = dinov2_df[dinov2_df['segment_id'] == seg_id]
    row_v3 = results_df[results_df['segment_id'] == seg_id]

    if row_vj.empty:
        continue

    gt   = row_vj.iloc[0]['ground_truth']
    cap  = row_vj.iloc[0]['masked_caption']

    print(f"Segment : {seg_id}")
    print(f"Caption : {cap}")
    print(f"GT verb : {gt}")

    if not row_v2.empty:
        r = row_v2.iloc[0]
        print(f"  DINOv2  Top-1: {r['top1_pred']:<12} {'✓' if r['top1_correct'] else '✗'}  Top-5: {r['top5_preds']}")
    if not row_v3.empty:
        r = row_v3.iloc[0]
        print(f"  DINOv3  Top-1: {r['top1_pred']:<12} {'✓' if r['top1_correct'] else '✗'}  Top-5: {r['top5_preds']}")
    if not row_vj.empty:
        r = row_vj.iloc[0]
        print(f"  V-JEPA2 Top-1: {r['top1_pred']:<12} {'✓' if r['top1_correct'] else '✗'}  Top-5: {r['top5_preds']}")
    print()

In [ ]:
# Get all segment_ids that have both DINOv2 and DINOv3 results
dinov2_segments = set(dinov2_df['segment_id'])
dinov3_segments = set(results_df['segment_id'])
common_segments = dinov2_segments & dinov3_segments

# Get the video_ids for those segments
common_df = df_val[df_val['segment_id'].isin(common_segments)].reset_index(drop=True)
common_video_ids = sorted(common_df['video_id'].unique())

print(f"DINOv2 segments   : {len(dinov2_segments):,}")
print(f"DINOv3 segments   : {len(dinov3_segments):,}")
print(f"Common segments   : {len(common_segments):,}")
print(f"Common video IDs  : {len(common_video_ids):,}")
print(f"\nVideo IDs:")
for v in common_video_ids:
    n_segs = common_df[common_df['video_id'] == v].shape[0]
    print(f"  {v}  ({n_segs} segments)")

# Save to file so you can load it in the V-JEPA2 notebook
import json
with open('/home/ubuntu/youcook2-videos/processed/common_video_ids.json', 'w') as f:
    json.dump({
        'video_ids'   : common_video_ids,
        'segment_ids' : list(common_segments)
    }, f, indent=2)
print(f"\nSaved to: /home/ubuntu/youcook2-videos/processed/common_video_ids.json")

In [ ]:
import pandas as pd

# Fixed: was loading the interrupted results_checkpoint.jsonl (566 rows). Now loads
# the completed full-val run — see the fix note in the cell above.
with open('/home/ubuntu/youcook2-videos/processed/common_video_ids.json') as f:
    common = json.load(f)
common_segment_ids = set(common['segment_ids'])

dinov2_df = pd.read_csv(os.path.join(PATHS['processed'], 'dinov2_zeroshot_fullval_fixed.csv'))

# Filter to common segments
dinov2_common = dinov2_df[dinov2_df['segment_id'].isin(common_segment_ids)]

top1 = dinov2_common['top1_correct'].mean()
top5 = dinov2_common['top5_correct'].mean()

print(f"=== DINOv2 + DinoTXT ===")
print(f"  Segments evaluated : {len(dinov2_common):,}")
print(f"  Val Top-1          : {top1:.4f}  ({top1*100:.2f}%)")
print(f"  Val Top-5          : {top5:.4f}  ({top5*100:.2f}%)")

In [ ]:
import json, pandas as pd

with open('/home/ubuntu/youcook2-videos/processed/common_video_ids.json') as f:
    common = json.load(f)
common_segment_ids = set(common['segment_ids'])

# results_df is the DINOv3 evaluation already run in this notebook
dinov3_common = results_df[results_df['segment_id'].isin(common_segment_ids)]

top1 = dinov3_common['top1_correct'].mean()
top5 = dinov3_common['top5_correct'].mean()

print(f"=== DINOv3 + DinoTXT ===")
print(f"  Segments evaluated : {len(dinov3_common):,}")
print(f"  Val Top-1          : {top1:.4f}  ({top1*100:.2f}%)")
print(f"  Val Top-5          : {top5:.4f}  ({top5*100:.2f}%)")

In [ ]:
import json, os, torch
import pandas as pd
import torch.nn.functional as F
from tqdm.auto import tqdm

# ── Load common segment ids ────────────────────────────────────────────────────
with open('/home/ubuntu/youcook2-videos/processed/common_video_ids.json') as f:
    common = json.load(f)
common_segment_ids = set(common['segment_ids'])

# ── Load DINOv2 DinoTXT with V-JEPA2 projection ───────────────────────────────
import sys
DINOV2_REPO = '/home/ubuntu/youcook2-videos/repos/dinov2'
if DINOV2_REPO not in sys.path:
    sys.path.insert(0, DINOV2_REPO)

from dinov2.hub.dinotxt import dinov2_vitl14_reg4_dinotxt_tet1280d20h24l, get_tokenizer
import torch.nn as nn

dinotxt_vj = dinov2_vitl14_reg4_dinotxt_tet1280d20h24l().to(DEVICE).eval()
tokenizer_vj = get_tokenizer()
dinotxt_vj = dinotxt_vj.to(DEVICE).eval()

# Replace projection with V-JEPA2-compatible shape
dinotxt_vj.text_model.head.linear_projection = nn.Linear(1280, 1024, bias=False).to(DEVICE)

# Load trained V-JEPA2 projection weights
CKPT_DIR = '/home/ubuntu/youcook2-videos/checkpoints'
ckpt = torch.load(
    os.path.join(CKPT_DIR, 'vjepa2_dinotxt_v2_cont_epoch5.pt'),
    map_location=DEVICE
)
dinotxt_vj.text_model.head.linear_projection.load_state_dict(ckpt)

for param in dinotxt_vj.parameters():
    param.requires_grad = False
dinotxt_vj.eval()

print(f"✓ DINOv2 DinoTXT loaded with V-JEPA2 projection")
print(f"  Projection shape: {dinotxt_vj.text_model.head.linear_projection.weight.shape}")

# ── Load data (standardized on the clean CSV — was segments_verbs_timed.csv,
# which has 19 extra val segments not present in the DINOv2/DINOv3 clean val set) ──
with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

df_all  = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_clean.csv')
df_val  = df_all[df_all['subset'] == 'validation'].reset_index(drop=True)
df_eval = df_val[df_val['segment_id'].isin(common_segment_ids)].reset_index(drop=True)

verb_vocab = sorted(df_all['extracted_verb'].unique())
label2id   = {v: i for i, v in enumerate(verb_vocab)}

VJEPA2_EMB_DIR = os.path.join(PATHS['embeddings'], 'vjepa2_dinotxt')

print(f"Segments to evaluate: {len(df_eval):,}")

# ── Pre-tokenize bare verbs ────────────────────────────────────────────────────
verb_tokens = tokenizer_vj.tokenize(verb_vocab).to(DEVICE)

# ── Evaluation ────────────────────────────────────────────────────────────────
vjepa2_results = []
missing = 0

dinotxt_vj.eval()
with torch.no_grad():
    # pre-compute verb embeddings once
    verb_embs = dinotxt_vj.encode_text(verb_tokens, normalize=True)  # (282, 1024)

    for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="V-JEPA2 + DinoTXT"):
        emb_path = os.path.join(VJEPA2_EMB_DIR, f"{row['segment_id']}.pt")
        if not os.path.exists(emb_path):
            missing += 1
            continue

        video_emb = torch.load(emb_path, map_location=DEVICE)
        video_emb = F.normalize(video_emb.unsqueeze(0), dim=-1)

        sim          = (dinotxt_vj.logit_scale.exp() * video_emb @ verb_embs.T).squeeze(0)
        top5_indices = sim.topk(5).indices.tolist()
        top1_verb    = verb_vocab[top5_indices[0]]
        gt_verb      = row['extracted_verb']
        gt_idx       = label2id.get(gt_verb, -1)

        vjepa2_results.append({
            'segment_id'  : row['segment_id'],
            'ground_truth': gt_verb,
            'top1_pred'   : top1_verb,
            'top1_correct': top1_verb == gt_verb,
            'top5_correct': gt_idx in top5_indices,
        })

vjepa2_common = pd.DataFrame(vjepa2_results)
top1 = vjepa2_common['top1_correct'].mean()
top5 = vjepa2_common['top5_correct'].mean()

print(f"\n=== V-JEPA2 + DinoTXT ===")
print(f"  Segments evaluated : {len(vjepa2_common):,}")
print(f"  Missing embeddings : {missing:,}")
print(f"  Val Top-1          : {top1:.4f}  ({top1*100:.2f}%)")
print(f"  Val Top-5          : {top5:.4f}  ({top5*100:.2f}%)")

In [ ]:
dinov2_results_path = os.path.join(PATHS['processed'], 'nb5_dinotxt_zeroshot_results.csv')

if os.path.exists(dinov2_results_path):
    dinov2_df = pd.read_csv(dinov2_results_path)

    print("=" * 55)
    print(f"{'Model':<20} {'Top-1':>10} {'Top-5':>10}")
    print("=" * 55)

    for subset in ['training', 'validation']:
        print(f"\n  Subset: {subset}")
        print("-" * 55)

        sub_v2 = dinov2_df[dinov2_df['subset'] == subset]
        sub_v3 = results_df[results_df['subset'] == subset]

        if len(sub_v2) > 0:
            t1 = sub_v2['top1_correct'].mean()
            t5 = sub_v2['top5_correct'].mean()
            print(f"  {'DINOv2 + DinoTXT':<18} {t1*100:>9.2f}% {t5*100:>9.2f}%")

        if len(sub_v3) > 0:
            t1 = sub_v3['top1_correct'].mean()
            t5 = sub_v3['top5_correct'].mean()
            print(f"  {'DINOv3 + DinoTXT':<18} {t1*100:>9.2f}% {t5*100:>9.2f}%")

    print("=" * 55)
else:
    print(f"DINOv2 results not found at {dinov2_results_path}")
    print("Run NB5 (DINOv2 DinoTXT) first to enable comparison.")

## Per-verb accuracy breakdown

In [ ]:
per_verb = results_df.groupby('ground_truth').agg(
    count    = ('top1_correct', 'count'),
    top1_acc = ('top1_correct', 'mean'),
    top5_acc = ('top5_correct', 'mean'),
).sort_values('count', ascending=False)

print("Top 20 most frequent verbs (DINOv3 + DinoTXT):")
print(per_verb.head(20).to_string())

## Confusion analysis — most commonly confused verb pairs

In [ ]:
# Look at wrong predictions to find systematic confusions
wrong = results_df[~results_df['top1_correct']].copy()
confusion_pairs = wrong.groupby(['ground_truth', 'top1_pred']).size().reset_index(name='count')
confusion_pairs = confusion_pairs.sort_values('count', ascending=False)

print(f"Total wrong predictions : {len(wrong):,}")
print("\nTop 15 confused verb pairs (ground truth → predicted):")
print(confusion_pairs.head(15).to_string(index=False))

## Save results

In [ ]:
out_path = os.path.join(PATHS['processed'], 'nb5b_dinov3_dinotxt_zeroshot_results.csv')
results_df.to_csv(out_path, index=False)
print(f"Results saved → {out_path}")
print(f"Total predictions : {len(results_df):,}")